## 2.3 Embeddings: From Discrete Tokens to Continuous Representations

### 2.3.1 Why Do We Need Embeddings?

Tokens are discrete symbols. Neural networks operate on continuous vectors. We need a bridge.

**The Embedding Matrix:** $\mathbf{E} \in \mathbb{R}^{|\mathcal{V}| \times d}$

For token $x$ with vocabulary index $k$:

$$\mathbf{e}_x = \mathbf{E}[k, :] \in \mathbb{R}^d$$

This is just a lookup table — row $k$ of $\mathbf{E}$ is the embedding for the $k$-th token.

**Equivalently:** If $\mathbf{x}$ is a one-hot vector:

$$\mathbf{e}_x = \mathbf{E}^\top \mathbf{x}$$

### 2.3.2 The Information-Theoretic View

**Discrete tokens → Continuous representation**

A one-hot vector over $|\mathcal{V}|$ tokens carries exactly $\log_2 |\mathcal{V}|$ bits (it specifies one of $|\mathcal{V}|$ possibilities).

The embedding $\mathbf{e}_x \in \mathbb{R}^d$ is a continuous vector. In principle, it could carry infinite information. But in practice, noise and finite precision limit this.

**What does embedding space capture?**
- Similarity: Similar words should have similar embeddings
- Relationships: Analogies like king - man + woman ≈ queen
- Context: The embedding provides a starting point that the model refines

**No Information Loss (So Far):** The embedding lookup is deterministic. No information from the token is lost — it's just re-represented.

## 2.4 Positional Encoding: Adding Information About Structure

### 2.4.1 The Problem: Transformers Are Permutation Invariant

Attention (as we'll see) treats all positions symmetrically. Without position information:

$$\text{Attention}(\text{"dog bites man"}) = \text{Attention}(\text{"man bites dog"})$$

This is clearly wrong. Position matters for meaning.

### 2.4.2 The Solution: Add Position Information

The original transformer uses sinusoidal positional encodings:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

where $pos$ is the position (0, 1, 2, ...) and $i$ is the dimension index.

**The Combined Embedding:**

$$\mathbf{h}_t^{(0)} = \mathbf{e}_{x_t} + \mathbf{PE}_t$$

### 2.4.3 Why This Specific Form?

**Why sinusoids?**

1. **Bounded:** $\sin$ and $\cos$ are in $[-1, 1]$, so positions don't dominate
2. **Unique:** Each position gets a unique pattern
3. **Relative positions can be computed:** $PE_{pos+k}$ is a linear function of $PE_{pos}$
4. **Generalization:** Can extrapolate to positions not seen during training

**Proof of linear relationship:** For any fixed offset $k$:

$$PE_{pos+k} = \mathbf{M}_k \cdot PE_{pos}$$

where $\mathbf{M}_k$ is a rotation matrix. This comes from trigonometric identities:

$$\sin(A + B) = \sin A \cos B + \cos A \sin B$$
$$\cos(A + B) = \cos A \cos B - \sin A \sin B$$

### 2.4.4 The Entropy View

Without positional encoding, the model has less information about the input. It cannot distinguish "dog bites man" from "man bites dog". By adding positions:

- We **add information** about word order
- We **reduce entropy** in the model's uncertainty about input structure
- We enable the model to learn position-dependent patterns

**Alternative:** Modern models often use learned positional embeddings or relative positional encodings (RoPE, ALiBi). The principle is the same — inject position information.

## 2.5 The Attention Mechanism: Contextualizing Representations

This is the core innovation. Let's derive it from first principles.

### 2.5.1 The Problem Attention Solves

Given a sequence of representations $\mathbf{h}_1, \mathbf{h}_2, \ldots, \mathbf{h}_T$, we want to create new representations that incorporate context from the entire sequence.

**Why is this necessary?** The embedding of "bank" is the same whether it means "financial institution" or "river bank". We need context to disambiguate. The only way to get context is to look at other positions.

### 2.5.2 The Core Idea: Weighted Combination of Values

For each position $t$, we want to compute:

$$\mathbf{z}_t = \sum_{s=1}^{T} \alpha_{ts} \cdot \mathbf{v}_s$$

where:
- $\mathbf{v}_s$ is some "value" representation of position $s$
- $\alpha_{ts}$ is the attention weight (how much position $t$ attends to position $s$)
- $\sum_s \alpha_{ts} = 1$ (attention weights form a probability distribution)

**Interpretation:** The output at position $t$ is a weighted average of values from all positions, where the weights depend on relevance.

### 2.5.3 Queries, Keys, and Values: Why Three Projections?

Here's where people usually hand-wave. Let's be precise.

**The Three Roles:**

1. **Query (Q):** "What am I looking for?" — represents what this position needs
2. **Key (K):** "What do I have?" — represents what this position offers
3. **Value (V):** "What information do I provide?" — the actual content to retrieve

**The Projections:**

$$\mathbf{Q} = \mathbf{H} \mathbf{W}^Q, \quad \mathbf{K} = \mathbf{H} \mathbf{W}^K, \quad \mathbf{V} = \mathbf{H} \mathbf{W}^V$$

where:
- $\mathbf{H} \in \mathbb{R}^{T \times d}$ is the input (each row is one position)
- $\mathbf{W}^Q, \mathbf{W}^K \in \mathbb{R}^{d \times d_k}$ are query/key projections
- $\mathbf{W}^V \in \mathbb{R}^{d \times d_v}$ is the value projection

**Why can't we use just one or two projections?**

**Case 1: No projection (just use H directly)**

Attention scores would be $\mathbf{H}\mathbf{H}^\top$. This only captures symmetric similarity — if A is similar to B, then B is equally similar to A. But attention should be asymmetric: "the" might need to attend to "cat" (to know what article applies to), but "cat" might not care about "the".

**Case 2: Two projections (Q and K only, V = H)**

This limits what information can be extracted. The same representation must serve as both "what I have" and "what I provide". Separating V allows the model to extract different information for attention-weighted combination than for matching.

**Case 3: Three projections (Q, K, V)**

- Q and K determine the attention pattern (who looks at whom)
- V determines what content flows through that pattern
- Full flexibility: attention routing and content retrieval are independent

**A Database Analogy:**
- Query: Your search query
- Key: Index/tags on database entries
- Value: The actual content you retrieve

You match queries against keys, but you retrieve values.

### 2.5.4 The Attention Score and Softmax

**Computing Compatibility:**

$$\text{score}(t, s) = \mathbf{q}_t^\top \mathbf{k}_s = \langle \mathbf{q}_t, \mathbf{k}_s \rangle$$

This is the dot product between query $t$ and key $s$. Higher dot product = more relevant.

**Scaling:**

$$\text{score}(t, s) = \frac{\mathbf{q}_t^\top \mathbf{k}_s}{\sqrt{d_k}}$$

**Why $\sqrt{d_k}$?** 

Assume $\mathbf{q}$ and $\mathbf{k}$ have components with mean 0 and variance 1. Their dot product:

$$\mathbf{q}^\top \mathbf{k} = \sum_{i=1}^{d_k} q_i k_i$$

Each $q_i k_i$ has mean 0 and variance 1 (product of independent unit-variance variables). Sum of $d_k$ such terms has variance $d_k$.

So $\mathbf{q}^\top \mathbf{k}$ has standard deviation $\sqrt{d_k}$.

If $d_k = 512$, the dot products have std $\approx 22.6$. Softmax on values with this spread gives nearly one-hot outputs (one weight near 1, rest near 0). This leads to:
- Vanishing gradients
- Inability to attend to multiple positions

Dividing by $\sqrt{d_k}$ normalizes to unit variance, keeping softmax in a reasonable regime.

**Converting to Weights:**

$$\alpha_{ts} = \text{softmax}_s\left(\frac{\mathbf{q}_t^\top \mathbf{k}_s}{\sqrt{d_k}}\right) = \frac{\exp(\mathbf{q}_t^\top \mathbf{k}_s / \sqrt{d_k})}{\sum_{s'} \exp(\mathbf{q}_t^\top \mathbf{k}_{s'} / \sqrt{d_k})}$$

### 2.5.5 The Full Attention Formula

Putting it all together:

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right) \mathbf{V}$$

In matrix form:
- $\mathbf{Q} \in \mathbb{R}^{T \times d_k}$
- $\mathbf{K} \in \mathbb{R}^{T \times d_k}$
- $\mathbf{V} \in \mathbb{R}^{T \times d_v}$
- $\mathbf{Q}\mathbf{K}^\top \in \mathbb{R}^{T \times T}$ (attention scores)
- Softmax applied row-wise
- Output $\in \mathbb{R}^{T \times d_v}$

### 2.5.6 The Entropy Interpretation of Attention

The attention weights $\alpha_{ts}$ form a probability distribution over positions for each query position $t$.

**High entropy attention:** Weights spread across many positions — position $t$ is gathering broad context.

**Low entropy attention:** Weights concentrated on few positions — position $t$ is focusing on specific relevant positions.

The model learns when to use diffuse vs. focused attention based on what reduces prediction error.

**In training:** The model learns Q, K, V projections such that attending to the right positions (those with relevant information) improves the categorical prediction at the end. This implicitly learns to route information where it's needed.

## 2.6 Multi-Head Attention: Why Multiple Heads?

### 2.6.1 The Limitation of Single-Head Attention

With one attention head, each position can only compute one weighted combination of values. But language has multiple types of relationships:
- Syntactic (subject-verb agreement)
- Semantic (word meanings)
- Positional (nearby words)
- Long-range (pronouns and antecedents)

One attention pattern can't capture all of these simultaneously.

### 2.6.2 Multi-Head Attention

**The Idea:** Run multiple attention operations in parallel, each with its own Q, K, V projections.

$$\text{head}_i = \text{Attention}(\mathbf{H}\mathbf{W}_i^Q, \mathbf{H}\mathbf{W}_i^K, \mathbf{H}\mathbf{W}_i^V)$$

$$\text{MultiHead}(\mathbf{H}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \mathbf{W}^O$$

where:
- $h$ is the number of heads
- $\mathbf{W}_i^Q, \mathbf{W}_i^K \in \mathbb{R}^{d \times d_k}$
- $\mathbf{W}_i^V \in \mathbb{R}^{d \times d_v}$
- $\mathbf{W}^O \in \mathbb{R}^{hd_v \times d}$ projects concatenated heads back to model dimension
- Typically $d_k = d_v = d/h$ (split dimension across heads)

### 2.6.3 Why Does This Help?

**Different heads learn different patterns:**
- Head 1 might learn syntactic dependencies
- Head 2 might learn positional locality
- Head 3 might learn semantic similarity
- etc.

**Information capacity:** With $h$ heads, the model can maintain $h$ different attention patterns simultaneously.

**Diminishing Returns:** There's a point where adding more heads doesn't help because:
1. Per-head dimension $d_k = d/h$ becomes too small
2. All useful attention patterns are already captured
3. Heads become redundant

Empirically, 8-16 heads work well for most models.

## 2.7 Masked Self-Attention: Causality for Autoregressive Models

### 2.7.1 The Problem with Full Attention

In language modeling, we predict $x_t$ given $x_{<t}$. If position $t$ can attend to position $s > t$, we're cheating — using future information.

During training, we process entire sequences in parallel. Without masking, the model could trivially copy the answer from position $t+1$ when predicting position $t$.

### 2.7.2 The Causal Mask

We modify attention scores before softmax:

$$\text{MaskedAttention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}} + \mathbf{M}\right) \mathbf{V}$$

where the mask $\mathbf{M} \in \mathbb{R}^{T \times T}$:

$$M_{ts} = \begin{cases} 0 & \text{if } s \leq t \\ -\infty & \text{if } s > t \end{cases}$$

**Why $-\infty$?**

$$\text{softmax}(\ldots, -\infty, \ldots)_k = \frac{e^{-\infty}}{Z} = \frac{0}{Z} = 0$$

Positions with $-\infty$ scores get zero attention weight after softmax.

### 2.7.3 The Resulting Pattern

The attention matrix becomes **lower triangular**:

```
Position:  1    2    3    4    5
        ┌────────────────────────┐
     1  │ ✓    ✗    ✗    ✗    ✗  │
     2  │ ✓    ✓    ✗    ✗    ✗  │
     3  │ ✓    ✓    ✓    ✗    ✗  │
     4  │ ✓    ✓    ✓    ✓    ✗  │
     5  │ ✓    ✓    ✓    ✓    ✓  │
        └────────────────────────┘
```

Position $t$ can only attend to positions $1, 2, \ldots, t$.

**This enforces autoregressive factorization:**

$$P(x_1, \ldots, x_T) = \prod_{t=1}^{T} P(x_t | x_1, \ldots, x_{t-1})$$

## 2.8 Feed-Forward Networks: Position-Wise Processing

### 2.8.1 Structure

After attention, each position passes through the same feed-forward network independently:

$$\text{FFN}(\mathbf{x}) = \text{ReLU}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

Or with GeLU (more common now):

$$\text{FFN}(\mathbf{x}) = \text{GeLU}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

where:
- $\mathbf{W}_1 \in \mathbb{R}^{d \times d_{ff}}$
- $\mathbf{W}_2 \in \mathbb{R}^{d_{ff} \times d}$
- Typically $d_{ff} = 4d$ (expansion factor of 4)

### 2.8.2 Why Do We Need FFN?

Attention is **linear** with respect to values. The attention output is a weighted sum of value vectors:

$$\mathbf{z}_t = \sum_s \alpha_{ts} \mathbf{v}_s$$

This limits expressiveness. FFN adds **non-linearity** at each position.

**Interpretation:** 
- Attention = communication between positions (information routing)
- FFN = local computation (information processing)

**Parameter Count:** In a standard transformer, FFN contains about 2/3 of the parameters per layer:
- Attention: $4d^2$ parameters (Q, K, V, O projections)
- FFN: $8d^2$ parameters ($d \times 4d + 4d \times d$)

So attention is only about 1/3 of the weights, not the whole story.

### 2.8.3 Modern Variants: Gated FFN

Many modern models use gated variants like SwiGLU:

$$\text{SwiGLU}(\mathbf{x}) = (\text{Swish}(\mathbf{x}\mathbf{W}_1) \odot (\mathbf{x}\mathbf{V})) \mathbf{W}_2$$

where $\text{Swish}(x) = x \cdot \sigma(x)$ and $\odot$ is element-wise multiplication.

This adds more expressiveness with minimal overhead.

## 2.9 Residual Connections and Layer Normalization

### 2.9.1 Residual Connections

Each sub-layer (attention, FFN) has a residual connection:

$$\mathbf{h}^{(l)} = \mathbf{h}^{(l-1)} + \text{SubLayer}(\mathbf{h}^{(l-1)})$$

**Why residuals?**

1. **Gradient flow:** Without residuals, gradients must flow through every layer. With residuals, they have a direct path:
   $$\frac{\partial \mathbf{h}^{(L)}}{\partial \mathbf{h}^{(0)}} = \mathbf{I} + \text{(other terms)}$$
   The identity component prevents vanishing gradients.

2. **Feature refinement:** Each layer can *add* information rather than replace it. The model learns incremental updates.

### 2.9.2 Layer Normalization

$$\text{LayerNorm}(\mathbf{x}) = \gamma \odot \frac{\mathbf{x} - \mu}{\sigma + \epsilon} + \beta$$

where:
- $\mu = \frac{1}{d}\sum_{i=1}^d x_i$ (mean across features)
- $\sigma = \sqrt{\frac{1}{d}\sum_{i=1}^d (x_i - \mu)^2}$ (std across features)
- $\gamma, \beta \in \mathbb{R}^d$ are learned parameters
- $\epsilon$ is a small constant for numerical stability

**Why normalize?**
- Stabilizes training by keeping activations in a reasonable range
- Reduces internal covariate shift
- Enables higher learning rates

**Pre-norm vs Post-norm:**

Original transformer (post-norm):
$$\mathbf{h}^{(l)} = \text{LayerNorm}(\mathbf{h}^{(l-1)} + \text{SubLayer}(\mathbf{h}^{(l-1)}))$$

Modern transformers (pre-norm):
$$\mathbf{h}^{(l)} = \mathbf{h}^{(l-1)} + \text{SubLayer}(\text{LayerNorm}(\mathbf{h}^{(l-1)}))$$

Pre-norm is more stable for deep models.

## 2.10 The Output Layer: Back to Token Probabilities

### 2.10.1 From Hidden State to Logits

After $L$ layers, we have final hidden states $\mathbf{h}_t^{(L)} \in \mathbb{R}^d$ for each position.

To predict the next token at position $t$, we project to vocabulary size:

$$\mathbf{z}_t = \mathbf{h}_t^{(L)} \mathbf{W}_{\text{out}} + \mathbf{b}_{\text{out}}$$

where $\mathbf{W}_{\text{out}} \in \mathbb{R}^{d \times |\mathcal{V}|}$ and $\mathbf{z}_t \in \mathbb{R}^{|\mathcal{V}|}$.

### 2.10.2 Weight Tying

A common trick: **tie the output projection to the input embeddings**:

$$\mathbf{W}_{\text{out}} = \mathbf{E}^\top$$

where $\mathbf{E}$ is the embedding matrix.

**Why does this work?**
- Reduces parameters by $d \times |\mathcal{V}|$ 
- Makes sense: tokens that are similar in embedding space should have similar output logits
- Empirically works well, especially for smaller models

### 2.10.3 Softmax and Cross-Entropy

$$\hat{p}_k = \text{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

Loss for position $t$ with true token $x_t = k^*$:

$$\mathcal{L}_t = -\log \hat{p}_{k^*} = -z_{k^*} + \log \sum_j e^{z_j}$$

This is often computed using `log_softmax` followed by `nll_loss` for numerical stability.

---

# Part 3: The Entropy Reduction Pipeline

Let's trace how entropy decreases through the network.

## 3.1 Initial State: High Entropy

The input is a sequence of tokens. Before processing:
- The model's belief about the next token is nearly uniform (high entropy)
- No context has been incorporated

## 3.2 Embeddings: Representing the Raw Signal

Token embeddings provide a dense representation of each token in isolation. At this stage:
- Each position knows only its own token
- No cross-position information
- Still high uncertainty about predictions

## 3.3 Attention: Information Gathering

Attention allows positions to share information:
- Query-key matching identifies relevant context
- Value aggregation collects useful information
- Entropy decreases as context narrows possibilities

**Example:** For "The cat sat on the ___":
- Without context: Could be anything
- After attention: "mat", "floor", "couch" become likely
- Entropy of prediction decreases

## 3.4 FFN: Information Processing

FFN transforms the contextualized representations:
- Non-linear processing extracts higher-order features
- Position-wise computation refines predictions
- Combination with residual preserves useful information

## 3.5 Layer Stacking: Iterative Refinement

Each layer:
1. Gathers more context (attention)
2. Processes that context (FFN)
3. Adds refinements via residual

**Deeper = More refined predictions**

Early layers: Local syntax, simple patterns
Middle layers: Semantic relationships
Later layers: Complex reasoning, long-range dependencies

## 3.6 Final Output: Low Entropy Predictions

At the end:
- The model has extracted all learnable patterns
- Output distribution concentrates on likely tokens
- Entropy is minimized (conditional on what's learnable from data)

**The irreducible entropy** = the inherent unpredictability of language. Even a perfect model can't predict with certainty because language isn't deterministic.

---

# Part 4: Modern Architectural Choices

## 4.1 Encoder-Decoder vs. Decoder-Only

### 4.1.1 Original Transformer: Encoder-Decoder

The original "Attention is All You Need" paper targeted **machine translation**:
- **Encoder:** Processes source sentence (bidirectional attention)
- **Decoder:** Generates target sentence (causal attention)
- **Cross-attention:** Decoder attends to encoder outputs

$$\text{CrossAttention}(\mathbf{Q}_{\text{decoder}}, \mathbf{K}_{\text{encoder}}, \mathbf{V}_{\text{encoder}})$$

Queries come from decoder, keys and values from encoder.

### 4.1.2 The Shift to Decoder-Only

Modern LLMs (GPT series, Llama, Claude) are **decoder-only**:
- Single stack of layers with causal masking
- No encoder, no cross-attention
- Input is the prompt; output is the continuation

**Why the shift?**

1. **Simpler architecture:** One model type for everything
2. **Unified objective:** Just predict next token
3. **Flexibility:** Same model does translation, QA, summarization by appropriate prompting
4. **Scaling:** Fewer architectural choices, easier to scale

**Key insight:** An encoder-decoder doing translation can be reframed as a decoder-only model:
- Input: "[source sentence] → [target sentence so far]"
- Output: Next target token

The "understanding" of the source happens through self-attention over the concatenated input.

### 4.1.3 Self-Attention vs. Cross-Attention

**Self-attention:** Q, K, V all come from the same sequence
$$\text{SelfAttention}(\mathbf{H}) = \text{Attention}(\mathbf{H}\mathbf{W}^Q, \mathbf{H}\mathbf{W}^K, \mathbf{H}\mathbf{W}^V)$$

**Cross-attention:** Q from one sequence, K and V from another
$$\text{CrossAttention}(\mathbf{H}_1, \mathbf{H}_2) = \text{Attention}(\mathbf{H}_1\mathbf{W}^Q, \mathbf{H}_2\mathbf{W}^K, \mathbf{H}_2\mathbf{W}^V)$$

Cross-attention is still used in:
- Multi-modal models (image + text)
- Retrieval-augmented generation
- Some specialized architectures

## 4.2 Attention Dimension and Context Length

### 4.2.1 The Quadratic Bottleneck

Standard attention has $O(T^2)$ complexity:
- $\mathbf{Q}\mathbf{K}^\top$ computes $T \times T$ matrix
- Storing attention weights: $O(T^2)$ memory
- For $T = 100,000$, that's 10 billion elements per layer per head

### 4.2.2 Why Not Just n × n Where n is Maximum Sequence?

If we fixed attention size to max sequence length $n_{\max}$:
- Shorter sequences would waste computation
- Memory doesn't scale with actual input
- The actual bottleneck is the quadratic growth with *actual* $T$

### 4.2.3 Solutions: Efficient Attention

**Flash Attention:** Same computation, better memory access patterns. Uses tiling to reduce memory IO. Still $O(T^2)$ compute but much faster in practice.

**Linear Attention:** Replace softmax with linear kernels:
$$\text{LinearAttention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \phi(\mathbf{Q})(\phi(\mathbf{K})^\top \mathbf{V})$$

where $\phi$ is a feature map. This is $O(T)$ because we can compute $(\phi(\mathbf{K})^\top \mathbf{V})$ first as a $d_k \times d_v$ matrix.

**Sparse Attention:** Only compute attention for a subset of positions (local windows, strided patterns).

**State-Space Models (Mamba, etc.):** Alternative architectures that don't use attention at all.

## 4.3 Low-Rank Factorization of Projections

### 4.3.1 The Observation

Large matrices like $\mathbf{W}^V$ have $d^2$ parameters. Often, these matrices are approximately low-rank — most of their "action" happens in a lower-dimensional subspace.

### 4.3.2 Factorization

Instead of $\mathbf{W} \in \mathbb{R}^{d \times d}$, use:

$$\mathbf{W} \approx \mathbf{W}_{\text{down}} \mathbf{W}_{\text{up}}$$

where $\mathbf{W}_{\text{down}} \in \mathbb{R}^{d \times r}$ and $\mathbf{W}_{\text{up}} \in \mathbb{R}^{r \times d}$ with $r \ll d$.

**Parameter reduction:** $d^2 \rightarrow 2dr$

**Why does this work?**
- Neural network weights often have low intrinsic rank
- The approximation is learned, not imposed post-hoc
- Models like LoRA exploit this for efficient fine-tuning

### 4.3.3 Grouped Query Attention (GQA)

Instead of factorizing weight matrices, share key-value heads:
- All heads share the same K and V projections
- Each head has its own Q projection
- Reduces KV cache size for inference

## 4.4 Why Pack Everything Into Big Matrices?

### 4.4.1 The QKV Pattern

You'll often see:

$$\mathbf{W}^{QKV} = [\mathbf{W}^Q; \mathbf{W}^K; \mathbf{W}^V] \in \mathbb{R}^{d \times 3d}$$

Then:
$$[\mathbf{Q}, \mathbf{K}, \mathbf{V}] = \mathbf{H} \mathbf{W}^{QKV}$$

One matrix multiplication instead of three.

### 4.4.2 Why? Hardware Efficiency

**GPU/TPU optimization:**
- Large matrix multiplications are highly optimized
- Memory bandwidth is often the bottleneck
- One big matmul has better memory access patterns than three small ones

**Kernel fusion:**
- Combining operations reduces memory round-trips
- Intermediate results stay in fast cache

**Parallelism:**
- Larger operations expose more parallelism
- Better utilization of thousands of GPU cores

This is why practical code often looks different from the mathematical formulation — it's the same computation, restructured for hardware.

---

# Part 5: Implementation Details

## 5.1 Complete Single-Head Attention

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SingleHeadAttention(nn.Module):
    """
    Single-head self-attention with causal masking.
    
    This implements the core attention operation:
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k) + mask) V
    """
    
    def __init__(self, d_model: int, d_k: int = None, d_v: int = None):
        super().__init__()
        
        # Default: d_k = d_v = d_model (full dimension)
        self.d_k = d_k if d_k is not None else d_model
        self.d_v = d_v if d_v is not None else d_model
        self.d_model = d_model
        
        # Scaling factor: prevents softmax saturation
        # Derived from variance analysis of dot products
        self.scale = 1.0 / math.sqrt(self.d_k)
        
        # Linear projections
        # W^Q, W^K: project to space where relevance is measured
        # W^V: project to space of content to retrieve
        self.W_Q = nn.Linear(d_model, self.d_k, bias=False)
        self.W_K = nn.Linear(d_model, self.d_k, bias=False)
        self.W_V = nn.Linear(d_model, self.d_v, bias=False)
        
        # Output projection (if d_v != d_model)
        if self.d_v != d_model:
            self.W_O = nn.Linear(self.d_v, d_model, bias=False)
        else:
            self.W_O = nn.Identity()
    
    def forward(self, x: torch.Tensor, causal_mask: bool = True) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [batch, seq_len, d_model]
            causal_mask: If True, apply causal masking (for autoregressive models)
        
        Returns:
            Output tensor of shape [batch, seq_len, d_model]
        """
        batch_size, seq_len, _ = x.shape
        
        # Step 1: Compute Q, K, V projections
        # Q: "what am I looking for?"
        # K: "what do I have to offer?"
        # V: "what content do I provide if selected?"
        Q = self.W_Q(x)  # [batch, seq_len, d_k]
        K = self.W_K(x)  # [batch, seq_len, d_k]
        V = self.W_V(x)  # [batch, seq_len, d_v]
        
        # Step 2: Compute attention scores
        # scores[i,j] = how much position i should attend to position j
        # QK^T: [batch, seq_len, d_k] @ [batch, d_k, seq_len] = [batch, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1))  # [batch, seq_len, seq_len]
        
        # Step 3: Scale by sqrt(d_k)
        # Without scaling: if d_k=512, dot products have std≈22
        # Softmax on such large values → nearly one-hot → vanishing gradients
        scores = scores * self.scale
        
        # Step 4: Apply causal mask (if autoregressive)
        if causal_mask:
            # Create lower triangular mask
            # mask[i,j] = 0 if j <= i (can attend), -inf if j > i (cannot attend)
            mask = torch.triu(
                torch.ones(seq_len, seq_len, device=x.device) * float('-inf'),
                diagonal=1
            )
            scores = scores + mask
        
        # Step 5: Softmax to get attention weights
        # Each row sums to 1 (probability distribution over keys)
        # After masking, future positions get exp(-inf) = 0 weight
        attn_weights = F.softmax(scores, dim=-1)  # [batch, seq_len, seq_len]
        
        # Step 6: Apply attention weights to values
        # output[i] = weighted sum of all V[j] where weight = attn_weights[i,j]
        output = torch.matmul(attn_weights, V)  # [batch, seq_len, d_v]
        
        # Step 7: Project back to d_model if necessary
        output = self.W_O(output)  # [batch, seq_len, d_model]
        
        return output


# Example usage and verification
if __name__ == "__main__":
    # Parameters
    batch_size = 2
    seq_len = 4
    d_model = 8
    
    # Create module
    attn = SingleHeadAttention(d_model=d_model)
    
    # Random input
    x = torch.randn(batch_size, seq_len, d_model)
    
    # Forward pass
    out = attn(x, causal_mask=True)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {out.shape}")
    
    # Verify causality: change future tokens, output at t should not change
    x_modified = x.clone()
    x_modified[:, -1, :] = torch.randn(batch_size, d_model)  # Change last position
    out_modified = attn(x_modified, causal_mask=True)
    
    # Output at positions 0,1,2 should be identical
    print(f"Output unchanged at pos 0: {torch.allclose(out[:, 0], out_modified[:, 0])}")
    print(f"Output unchanged at pos 1: {torch.allclose(out[:, 1], out_modified[:, 1])}")
    print(f"Output unchanged at pos 2: {torch.allclose(out[:, 2], out_modified[:, 2])}")
```

## 5.2 Multi-Head Attention

```python
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention.
    
    Multiple attention heads allow the model to jointly attend to
    information from different representation subspaces.
    """
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head
        self.scale = 1.0 / math.sqrt(self.d_k)
        
        # Combined QKV projection for efficiency
        # Instead of 3 separate matmuls, we do one big one
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, causal_mask: bool = True) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [batch, seq_len, d_model]
            causal_mask: If True, apply causal masking
        
        Returns:
            Output tensor of shape [batch, seq_len, d_model]
        """
        batch_size, seq_len, _ = x.shape
        
        # Step 1: Combined QKV projection
        qkv = self.W_qkv(x)  # [batch, seq_len, 3*d_model]
        
        # Step 2: Split into Q, K, V
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.d_k)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, batch, heads, seq_len, d_k]
        Q, K, V = qkv[0], qkv[1], qkv[2]  # Each: [batch, heads, seq_len, d_k]
        
        # Step 3: Compute attention scores for all heads in parallel
        # [batch, heads, seq_len, d_k] @ [batch, heads, d_k, seq_len]
        # = [batch, heads, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        
        # Step 4: Apply causal mask
        if causal_mask:
            mask = torch.triu(
                torch.ones(seq_len, seq_len, device=x.device) * float('-inf'),
                diagonal=1
            )
            scores = scores + mask  # Broadcasting: [batch, heads, seq, seq]
        
        # Step 5: Softmax and dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Step 6: Apply attention to values
        # [batch, heads, seq_len, seq_len] @ [batch, heads, seq_len, d_k]
        # = [batch, heads, seq_len, d_k]
        output = torch.matmul(attn_weights, V)
        
        # Step 7: Concatenate heads and project
        # [batch, heads, seq_len, d_k] -> [batch, seq_len, heads*d_k]
        output = output.transpose(1, 2).reshape(batch_size, seq_len, self.d_model)
        output = self.W_o(output)
        
        return output
```

## 5.3 Complete Transformer Block

```python
class TransformerBlock(nn.Module):
    """
    A single transformer block with:
    - Multi-head self-attention
    - Feed-forward network
    - Residual connections
    - Layer normalization (pre-norm variant)
    """
    
    def __init__(
        self, 
        d_model: int, 
        num_heads: int, 
        d_ff: int = None,
        dropout: float = 0.1
    ):
        super().__init__()
        
        # Default FFN dimension is 4x model dimension
        d_ff = d_ff if d_ff is not None else 4 * d_model
        
        # Pre-normalization (more stable for deep networks)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Multi-head attention
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Feed-forward network
        # Linear(d_model -> d_ff) -> activation -> Linear(d_ff -> d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),  # Modern transformers prefer GELU over ReLU
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input of shape [batch, seq_len, d_model]
        
        Returns:
            Output of shape [batch, seq_len, d_model]
        """
        # Attention block with residual connection
        # x = x + Attention(LayerNorm(x))
        x = x + self.attention(self.norm1(x))
        
        # FFN block with residual connection
        # x = x + FFN(LayerNorm(x))
        x = x + self.ffn(self.norm2(x))
        
        return x
```

## 5.4 Complete Decoder-Only Language Model

```python
class DecoderOnlyLM(nn.Module):
    """
    Complete decoder-only language model.
    
    Architecture:
    1. Token embedding
    2. Positional encoding
    3. Stack of transformer blocks
    4. Final layer norm
    5. Output projection to vocabulary
    """
    
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 512,
        num_layers: int = 6,
        num_heads: int = 8,
        d_ff: int = 2048,
        max_seq_len: int = 1024,
        dropout: float = 0.1,
        tie_weights: bool = True  # Tie input/output embeddings
    ):
        super().__init__()
        
        self.d_model = d_model
        self.vocab_size = vocab_size
        
        # Token embedding: lookup table of size [vocab_size, d_model]
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional encoding: learned (simpler than sinusoidal, works well)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        
        # Embedding dropout
        self.embed_dropout = nn.Dropout(dropout)
        
        # Stack of transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Final layer normalization
        self.final_norm = nn.LayerNorm(d_model)
        
        # Output projection to vocabulary
        # If tie_weights, this shares parameters with token_embedding
        self.output_projection = nn.Linear(d_model, vocab_size, bias=False)
        
        if tie_weights:
            # Weight tying: output projection = transpose of input embedding
            self.output_projection.weight = self.token_embedding.weight
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights following GPT-2 conventions."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(
        self, 
        input_ids: torch.Tensor, 
        targets: torch.Tensor = None
    ) -> tuple:
        """
        Args:
            input_ids: Token indices of shape [batch, seq_len]
            targets: Target token indices for loss computation (optional)
        
        Returns:
            logits: Output logits of shape [batch, seq_len, vocab_size]
            loss: Cross-entropy loss if targets provided, else None
        """
        batch_size, seq_len = input_ids.shape
        device = input_ids.device
        
        # Step 1: Token embeddings
        # Look up embedding vector for each input token
        tok_emb = self.token_embedding(input_ids)  # [batch, seq, d_model]
        
        # Step 2: Positional embeddings
        # Create position indices [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(seq_len, device=device)
        pos_emb = self.position_embedding(positions)  # [seq, d_model]
        
        # Step 3: Combine embeddings
        x = self.embed_dropout(tok_emb + pos_emb)
        
        # Step 4: Pass through transformer layers
        for layer in self.layers:
            x = layer(x)
        
        # Step 5: Final layer norm
        x = self.final_norm(x)
        
        # Step 6: Project to vocabulary
        logits = self.output_projection(x)  # [batch, seq, vocab_size]
        
        # Step 7: Compute loss if targets provided
        loss = None
        if targets is not None:
            # Reshape for cross-entropy
            # logits: [batch * seq, vocab_size]
            # targets: [batch * seq]
            loss = F.cross_entropy(
                logits.view(-1, self.vocab_size),
                targets.view(-1),
                ignore_index=-100  # Standard padding token to ignore
            )
        
        return logits, loss
    
    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int = None
    ) -> torch.Tensor:
        """
        Autoregressive generation.
        
        Args:
            input_ids: Initial tokens [batch, seq]
            max_new_tokens: Number of tokens to generate
            temperature: Sampling temperature (higher = more random)
            top_k: If set, only sample from top k tokens
        
        Returns:
            Generated sequence including input tokens
        """
        for _ in range(max_new_tokens):
            # Get predictions for current sequence
            logits, _ = self.forward(input_ids)
            
            # Focus on last position (next token prediction)
            logits = logits[:, -1, :] / temperature
            
            # Optional: top-k filtering
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            # Convert to probabilities
            probs = F.softmax(logits, dim=-1)
            
            # Sample from distribution
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append to sequence
            input_ids = torch.cat([input_ids, next_token], dim=1)
        
        return input_ids


# Example: Create a small model and test
if __name__ == "__main__":
    # Small model for testing
    model = DecoderOnlyLM(
        vocab_size=1000,
        d_model=128,
        num_layers=2,
        num_heads=4,
        d_ff=512,
        max_seq_len=64
    )
    
    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}")
    
    # Test forward pass
    batch_size = 4
    seq_len = 16
    x = torch.randint(0, 1000, (batch_size, seq_len))
    targets = torch.randint(0, 1000, (batch_size, seq_len))
    
    logits, loss = model(x, targets)
    print(f"Logits shape: {logits.shape}")
    print(f"Loss: {loss.item():.4f}")
    
    # Test generation
    prompt = torch.randint(0, 1000, (1, 5))
    generated = model.generate(prompt, max_new_tokens=10)
    print(f"Generated shape: {generated.shape}")
```

## 5.5 The Cross-Entropy Loss Computation: Explicit

```python
def cross_entropy_loss_explicit(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Explicit cross-entropy loss computation for educational purposes.
    
    This shows exactly what PyTorch's F.cross_entropy does.
    
    Args:
        logits: Raw model outputs [batch, seq_len, vocab_size]
        targets: Ground truth token indices [batch, seq_len]
    
    Returns:
        Scalar loss value
    """
    batch_size, seq_len, vocab_size = logits.shape
    
    # Flatten for easier computation
    logits_flat = logits.view(-1, vocab_size)  # [batch*seq, vocab]
    targets_flat = targets.view(-1)  # [batch*seq]
    
    # Step 1: Log-Softmax (numerically stable version)
    # log_softmax(z)_k = z_k - log(sum_j exp(z_j))
    #                  = z_k - logsumexp(z)
    
    # Compute logsumexp with numerical stability
    # logsumexp(z) = max(z) + log(sum_j exp(z_j - max(z)))
    max_logits = logits_flat.max(dim=-1, keepdim=True).values
    log_sum_exp = max_logits + torch.log(
        torch.exp(logits_flat - max_logits).sum(dim=-1, keepdim=True)
    )
    log_probs = logits_flat - log_sum_exp  # [batch*seq, vocab]
    
    # Step 2: Select log probability of correct class
    # This is the negative log likelihood
    # For each position, get log_prob[target]
    num_samples = targets_flat.shape[0]
    indices = torch.arange(num_samples, device=targets_flat.device)
    nll = -log_probs[indices, targets_flat]  # [batch*seq]
    
    # Step 3: Average over all positions
    loss = nll.mean()
    
    return loss


# Verify it matches PyTorch
if __name__ == "__main__":
    logits = torch.randn(4, 16, 1000)
    targets = torch.randint(0, 1000, (4, 16))
    
    # Our implementation
    loss_ours = cross_entropy_loss_explicit(logits, targets)
    
    # PyTorch implementation
    loss_pytorch = F.cross_entropy(
        logits.view(-1, 1000), 
        targets.view(-1)
    )
    
    print(f"Our loss: {loss_ours.item():.6f}")
    print(f"PyTorch loss: {loss_pytorch.item():.6f}")
    print(f"Match: {torch.allclose(loss_ours, loss_pytorch)}")
```

---

# Part 6: Summary and Key Insights

## 6.1 The Big Picture

1. **Transformers do MLE on categorical distributions** — nothing more, nothing less
2. **Cross-entropy loss is negative log-likelihood** for one-hot targets
3. **Every component reduces entropy** of the prediction:
   - BPE: Efficient information encoding
   - Embeddings: Dense representation of tokens
   - Positional encoding: Add structural information
   - Attention: Gather relevant context
   - FFN: Process and transform information
   - Layer stacking: Iterative refinement

## 6.2 The Entropy Thread

$$\text{Raw input} \xrightarrow{\text{BPE}} \text{Tokens} \xrightarrow{\text{Embed}} \text{Vectors} \xrightarrow{\text{+Position}} \text{Positioned} \xrightarrow{\text{Layers}} \text{Refined} \xrightarrow{\text{Softmax}} \text{Distribution}$$

At each step, we either:
- Add information (reducing entropy)
- Transform information (preserving/concentrating entropy)
- Combine information (contextualizing, reducing conditional entropy)

## 6.3 Why Transformers Work

1. **Attention is a learned information routing mechanism** — it decides what context is relevant
2. **Residual connections preserve information** — no forced compression
3. **Layer depth = iterative refinement** — early layers handle simple patterns, later layers handle complex reasoning
4. **Scale matters** — more parameters = more patterns = lower entropy predictions

## 6.4 Common Misconceptions

1. **"Attention is all you need"** — Actually, attention is only ~1/3 of parameters. FFN matters too.
2. **"Transformers understand language"** — They minimize cross-entropy. Understanding is a useful metaphor, not a technical claim.
3. **"More parameters always helps"** — There are diminishing returns and convergence points.
4. **"The architecture is magical"** — Every component has a principled justification in terms of the learning objective.

---

# Appendix A: Mathematical Prerequisites

## A.1 The Softmax Function

**Definition:**

$$\text{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

**Properties:**
1. Outputs sum to 1: $\sum_i \text{softmax}(\mathbf{z})_i = 1$
2. All outputs positive: $\text{softmax}(\mathbf{z})_i > 0$
3. Translation invariant: $\text{softmax}(\mathbf{z} + c) = \text{softmax}(\mathbf{z})$
4. Temperature scaling: $\text{softmax}(\mathbf{z}/\tau)$ — lower $\tau$ = sharper distribution

**Gradient:**

$$\frac{\partial \text{softmax}(\mathbf{z})_i}{\partial z_j} = \text{softmax}(\mathbf{z})_i (\delta_{ij} - \text{softmax}(\mathbf{z})_j)$$

where $\delta_{ij}$ is the Kronecker delta.

## A.2 The Log-Sum-Exp Trick

Computing $\log \sum_j e^{z_j}$ directly causes overflow for large $z_j$.

**Solution:**

$$\log \sum_j e^{z_j} = m + \log \sum_j e^{z_j - m}$$

where $m = \max_j z_j$. Now $z_j - m \leq 0$, so $e^{z_j - m} \leq 1$.

## A.3 KL Divergence

**Definition:**

$$D_{KL}(p \| q) = \sum_k p_k \log \frac{p_k}{q_k} = \sum_k p_k \log p_k - \sum_k p_k \log q_k$$

**Properties:**
1. Non-negative: $D_{KL}(p \| q) \geq 0$
2. Zero iff $p = q$
3. Not symmetric: $D_{KL}(p \| q) \neq D_{KL}(q \| p)$
4. Not a metric (doesn't satisfy triangle inequality)

---

# Appendix B: Notation Reference

| Symbol | Meaning |
|--------|---------|
| $\mathcal{V}$ | Vocabulary (set of all tokens) |
| $\|\mathcal{V}\|$ | Vocabulary size |
| $T$ | Sequence length |
| $d$ or $d_{\text{model}}$ | Model dimension |
| $d_k$ | Key/Query dimension |
| $d_v$ | Value dimension |
| $d_{ff}$ | Feed-forward hidden dimension |
| $h$ | Number of attention heads |
| $L$ | Number of layers |
| $\mathbf{E}$ | Embedding matrix |
| $\mathbf{W}^Q, \mathbf{W}^K, \mathbf{W}^V$ | Attention projections |
| $\mathbf{W}^O$ | Output projection |
| $\mathbf{h}^{(l)}$ | Hidden state at layer $l$ |
| $\alpha_{ts}$ | Attention weight (position $t$ → $s$) |
| $H(p)$ | Entropy of distribution $p$ |
| $H(p, q)$ | Cross-entropy between $p$ and $q$ |
| $D_{KL}(p \| q)$ | KL divergence from $q$ to $p$ |

---

*"The transformer is not magic. It's a very sophisticated function approximator that has been trained to minimize prediction error on human text. Every architectural choice exists to help that optimization. Once you see it this way, the mystique disappears and understanding begins."*